# 07 · Gold — Daily Account Balance

**Layer:** Gold &nbsp;|&nbsp; **Stage:** aggregation &nbsp;|&nbsp; **Target:** `banking_gold.account_balance_daily`

First of the two gold aggregates, running in parallel with
`08_gold_fraud_signal_summary`. Produces the table the retail-banking
dashboard reads: one row per account per day, carrying that day's movement
and the running closing balance carried forward from the opening balance on
the account dimension.

Uses a window function to accumulate the balance rather than a self-join, so
the whole thing stays a single shuffle.


In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("medallion-gold-balance").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "true")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
SILVER_DB = "banking_silver"
GOLD_DB   = "banking_gold"

txn = spark.table(f"{SILVER_DB}.txn_clean")
dim = spark.table(f"{SILVER_DB}.dim_account")
print(f"aggregating {txn.count()} transactions across {dim.count()} accounts")

In [ ]:
# Per-account, per-day movement.
daily = (
    txn.groupBy("account_id", "txn_date")
    .agg(
        F.count("*").alias("txn_count"),
        F.sum(F.when(F.col("direction") == "DEBIT", F.col("amount_inr")).otherwise(0.0)).alias("total_debit_inr"),
        F.sum(F.when(F.col("direction") == "CREDIT", F.col("amount_inr")).otherwise(0.0)).alias("total_credit_inr"),
        F.sum("signed_amount_inr").alias("net_movement_inr"),
        F.round(F.avg("amount_inr"), 2).alias("avg_txn_inr"),
        F.max("amount_inr").alias("max_txn_inr"),
        F.countDistinct("channel").alias("channels_used"),
        F.countDistinct("merchant").alias("distinct_merchants"),
    )
)

In [ ]:
# Running balance: opening balance on the dimension, plus the cumulative
# net movement up to and including each day.
running = Window.partitionBy("account_id").orderBy("txn_date") \
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)

balance_daily = (
    daily
    .join(
        dim.select(
            "account_id", "customer_id", "branch_code", "account_type",
            "currency", "opening_balance", "risk_category", "status",
        ),
        on="account_id", how="inner",
    )
    .withColumn("cumulative_movement_inr", F.round(F.sum("net_movement_inr").over(running), 2))
    .withColumn("closing_balance_inr",
                F.round(F.col("opening_balance") + F.col("cumulative_movement_inr"), 2))
    .withColumn("is_overdrawn", F.col("closing_balance_inr") < 0)
    .withColumn(
        "activity_tier",
        F.when(F.col("txn_count") >= 5, "HIGH")
         .when(F.col("txn_count") >= 2, "MEDIUM")
         .otherwise("LOW"),
    )
)

In [ ]:
account_balance_daily = (
    balance_daily
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .withColumn("_run_date", F.lit(RUN_DATE))
    .withColumn("_generated_at", F.current_timestamp())
    .select(
        "txn_date", "account_id", "customer_id", "branch_code", "account_type",
        "currency", "status", "risk_category",
        "txn_count", "channels_used", "distinct_merchants", "activity_tier",
        F.round("total_debit_inr", 2).alias("total_debit_inr"),
        F.round("total_credit_inr", 2).alias("total_credit_inr"),
        F.round("net_movement_inr", 2).alias("net_movement_inr"),
        "avg_txn_inr", "max_txn_inr",
        "opening_balance", "cumulative_movement_inr", "closing_balance_inr", "is_overdrawn",
        "_batch_id", "_run_date", "_generated_at",
    )
)

(
    account_balance_daily.write
    .mode("overwrite").format("parquet").partitionBy("txn_date")
    .saveAsTable(f"{GOLD_DB}.account_balance_daily")
)

out = spark.table(f"{GOLD_DB}.account_balance_daily")
print(f"wrote {GOLD_DB}.account_balance_daily  ({out.count()} rows)")

In [ ]:
out.orderBy(F.desc("net_movement_inr")).show(20, truncate=False)

In [ ]:
print("branch-level roll-up")

out.groupBy("branch_code").agg(
    F.countDistinct("account_id").alias("accounts"),
    F.sum("txn_count").alias("transactions"),
    F.round(F.sum("total_debit_inr"), 2).alias("debits_inr"),
    F.round(F.sum("total_credit_inr"), 2).alias("credits_inr"),
    F.round(F.sum("closing_balance_inr"), 2).alias("closing_inr"),
).orderBy(F.desc("debits_inr")).show(truncate=False)